# Aegis Voice Agent: Colab launcher

Starts the full stack from the repo on a Colab **L4 GPU**: Gemma 4 E4B (hears + answers), Indic-Mio (voice), the LiveKit agent worker, the dialer API (`POST /dial`) and the live dashboard.

**Before running**
1. Runtime -> Change runtime type -> **L4 GPU**.
2. Add the secrets from `.env.example` under the key icon and enable notebook access for each. Required: `TAILSCALE_AUTH_KEY`, `LIVEKIT_HOST`, `LIVEKIT_API_KEY`, `LIVEKIT_API_SECRET`, `TATA_PILOT_CLI`. Set `N8N_WEBHOOK_SECRET` if your n8n webhooks are public.
3. Optional: upload a 5-15 s reference voice as `/content/naina.wav`.
4. Run the cells top to bottom. Rerun the network cell after any runtime restart.

Architecture, API contract and limitations: see the repository README.


In [ ]:
# 0. Get the code from GitHub, then settings.
import os, subprocess
REPO_URL = "https://github.com/YOUR_GITHUB_USER/aegis-voice-agent"   # <- set to your repo
REPO = "/content/aegis-voice-agent"
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO], check=True)
SRC = f"{REPO}/src"
os.environ["AEGIS_DATA_DIR"] = "/content"        # logs, outbox, debug audio

ENGINE = "vllm"                                          # "vllm" or "llamacpp"
GEMMA_MODEL = "google/gemma-4-E4B-it-qat-mobile-ct"      # used when ENGINE = "vllm"
GGUF_REPO = "google/gemma-4-E4B-it-qat-q4_0-gguf"        # used when ENGINE = "llamacpp"
GGUF_QUANT = "q4_0"

# Must match provision.py and dial.py ("aegis-agent")
AGENT_NAME = "aegis-agent"

# Outbound dialer (n8n calls POST /dial). Keys/secrets live in Colab secrets, never here.
MAX_CONCURRENT_CALLS = 2        # rooms allowed at once (GPU is shared with Gemma + Mio); n8n gets HTTP 429 when full
CALL_WINDOW = "10:00-20:00"     # IST window in which /dial accepts calls; "" disables the check
SIP_TRUNK_ID = ""               # LiveKit outbound trunk id; "" = use the only trunk (errors if there are several)

# Reference voice clip (5-15 s wav). Leave "" to fall back to sample voice if naina.wav is not uploaded yet.
import os, subprocess
REF_VOICE = "/content/naina.wav" if os.path.exists("/content/naina.wav") else ""

# Voice delivery settings
SPEAKING_RATE = 1.05           # 1.0 = default; 0.85-0.95 = calmer pace
CLAUSE_PAUSE = 0.18            # seconds of silence between clauses
STYLE_HINDI = "happy"
STYLE_ENGLISH = "happy"

assert ENGINE in ("vllm", "llamacpp")
assert STYLE_HINDI in ("", "happy", "sad", "angry", "disgust", "fear", "surprise"), "STYLE_HINDI: not a Mio tag"
assert STYLE_ENGLISH in ("", "happy", "sad", "enunciated", "confused", "angry", "whisper"), "STYLE_ENGLISH: not a Mio tag"
gpu = subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader", shell=True,
                     capture_output=True, text=True).stdout.strip()
print("Gemma:", GEMMA_MODEL if ENGINE == "vllm" else f"{GGUF_REPO} ({GGUF_QUANT})", "on", ENGINE,
      "| voice:", REF_VOICE or "sample", "| GPU:", gpu or "NONE")
assert gpu, "No GPU. Runtime -> Change runtime type -> L4 GPU."


In [ ]:
# 3. Secrets and environment (rerun this cell after any runtime restart)
import glob
from google.colab import userdata

import getpass, secrets as _secrets

def secret(name, required=True, default=""):
    """Colab secret first. If it is missing and required, ask for it once (hidden input) instead of using a key stored in the notebook."""
    try:
        v = userdata.get(name)
    except Exception:
        v = None
    if v:
        return v
    if required:
        v = getpass.getpass(f"Colab secret {name} is not set. Paste it now (not stored; add it under the key icon to skip this prompt): ").strip()
        if not v:
            raise SystemExit(f"{name} is required.")
        return v
    return default

def dialer_key():
    try:
        v = userdata.get("DIALER_API_KEY")
    except Exception:
        v = None
    if v:
        return v
    v = _secrets.token_urlsafe(32)
    print("DIALER_API_KEY secret not set: generated one for this session only.")
    print("Put this in the n8n credential (Header Auth, name X-Api-Key), and save it as the Colab secret DIALER_API_KEY so it stays the same:")
    print("   ", v)
    return v

os.environ["LIVEKIT_HOST"] = secret("LIVEKIT_HOST")          # IPv4 of LiveKit on your tailnet
os.environ["LK_TARGET_IP"] = os.environ["LIVEKIT_HOST"]      # used by the bridge + LD_PRELOAD shim
os.environ["LIVEKIT_URL"] = secret("LIVEKIT_URL", required=False, default=f"ws://{os.environ['LIVEKIT_HOST']}:7880")
os.environ["PBX_HOST"] = secret("PBX_HOST", required=False)
os.environ["TATA_PILOT_CLI"] = secret("TATA_PILOT_CLI")      # caller-ID / pilot number of your SIP trunk
os.environ["LIVEKIT_API_KEY"] = secret("LIVEKIT_API_KEY")
os.environ["LIVEKIT_API_SECRET"] = secret("LIVEKIT_API_SECRET")
os.environ["DIALER_API_KEY"] = dialer_key()                       # n8n -> dialer, header X-Api-Key
os.environ["N8N_WEBHOOK_SECRET"] = secret("N8N_WEBHOOK_SECRET", required=False)  # agent -> n8n, header X-Api-Key
os.environ["N8N_WHATSAPP_WEBHOOK"] = secret("N8N_WHATSAPP_WEBHOOK", required=False, default=f"http://{os.environ['LIVEKIT_HOST']}:5678/webhook/send-b2b-link")
os.environ["N8N_CRM_UPDATE_WEBHOOK"] = secret("N8N_CRM_UPDATE_WEBHOOK", required=False, default=f"http://{os.environ['LIVEKIT_HOST']}:5678/webhook/update-twenty-crm")
os.environ["CALLBACK_ALLOWED_HOSTS"] = secret("CALLBACK_ALLOWED_HOSTS", required=False)
os.environ["MAX_CONCURRENT_CALLS"] = str(MAX_CONCURRENT_CALLS)
os.environ["CALL_WINDOW"] = CALL_WINDOW
os.environ["SIP_TRUNK_ID"] = SIP_TRUNK_ID

import ipaddress
from urllib.parse import urlparse
def _is_private(url):
    h = urlparse(url).hostname or ""
    if h == "localhost":
        return True
    try:
        ip = ipaddress.ip_address(h)
    except ValueError:
        return False   # a domain name such as n8n.example.com is public
    return ip.is_private or ip in ipaddress.ip_network("100.64.0.0/10")
_public = [u for u in (os.environ["N8N_WHATSAPP_WEBHOOK"], os.environ["N8N_CRM_UPDATE_WEBHOOK"]) if not _is_private(u)]
if _public and not os.environ["N8N_WEBHOOK_SECRET"]:
    raise SystemExit(f"Your n8n webhooks are public ({_public[0]}). Add the Colab secret N8N_WEBHOOK_SECRET and protect those "
                     "Webhook nodes with Header Auth (X-Api-Key), otherwise anyone can post fake call outcomes into your CRM.")
if not os.environ["N8N_WEBHOOK_SECRET"]:
    print("Note: N8N_WEBHOOK_SECRET not set, so webhooks to n8n are sent without an X-Api-Key header.")

try:
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    print("HF_TOKEN secret not available: fine unless a gated download is requested.")

BASE_LD = ":".join(p for p in os.environ.get("LD_LIBRARY_PATH", "").split(":") if p and "/dist-packages/nvidia/" not in p)
os.environ["LD_LIBRARY_PATH"] = BASE_LD
def clean_env(**extra):
    return dict(os.environ, LD_LIBRARY_PATH=BASE_LD, **extra)
os.environ["MPLBACKEND"] = "agg"
print("LiveKit URL:", os.environ["LIVEKIT_URL"])
print("n8n Webhooks:", os.environ["N8N_WHATSAPP_WEBHOOK"], "|", os.environ["N8N_CRM_UPDATE_WEBHOOK"])


In [ ]:
# 0a. Network: Tailscale (userspace) + local TCP bridge + LD_PRELOAD shim, so the agent can reach a private LiveKit host.
# Colab has no TUN device, so Tailscale only offers a SOCKS5 proxy. See infra/tailscale_bridge/ for the bridge and shim.
import time, threading
subprocess.run("curl -fsSL https://tailscale.com/install.sh | sh && apt-get install -qq -y gcc", shell=True, check=True)

subprocess.run(["pkill", "-9", "tailscaled"], stderr=subprocess.DEVNULL); time.sleep(1)
with open("/content/tailscaled.log", "wb") as ts_log:
    subprocess.Popen(["tailscaled", "--tun=userspace-networking", "--socks5-server=127.0.0.1:1055",
                      "--outbound-http-proxy-listen=127.0.0.1:1056"],
                     stdout=ts_log, stderr=ts_log, start_new_session=True)
time.sleep(3)

ts_key = secret("TAILSCALE_AUTH_KEY")     # ephemeral + tagged key
subprocess.run(["tailscale", "up", f"--authkey={ts_key}", "--hostname=colab-voice-worker", "--accept-routes"], check=True, timeout=90)
subprocess.run(["tailscale", "set", "--accept-routes"], check=True)
del ts_key

subprocess.run(["pkill", "-9", "-f", "ts_bridge.py"], stderr=subprocess.DEVNULL)
subprocess.Popen(["python3", "-u", f"{REPO}/infra/tailscale_bridge/ts_bridge.py"],
                 stdout=open("/content/bridge.log", "w"), stderr=subprocess.STDOUT, start_new_session=True,
                 env=dict(os.environ, LK_TARGET_IP=os.environ["LIVEKIT_HOST"]))
subprocess.run(["gcc", "-shared", "-fPIC", "-O2", f"{REPO}/infra/tailscale_bridge/lk_redirect.c",
                "-o", "/content/liblk_redirect.so", "-ldl"], check=True)

def _keep_alive():   # keeps the direct WireGuard path warm (avoids DERP relay latency)
    hosts = [h for h in (os.environ.get("PBX_HOST"), os.environ["LIVEKIT_HOST"]) if h]
    while True:
        for h in hosts:
            subprocess.run(["tailscale", "ping", "-c", "1", h], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        time.sleep(5)
threading.Thread(target=_keep_alive, daemon=True).start()

time.sleep(2)
print(subprocess.run(["tailscale", "ping", "-c", "3", os.environ["LIVEKIT_HOST"]], capture_output=True, text=True).stdout)
print(open("/content/bridge.log", errors="ignore").read())
if open("/content/bridge.log", errors="ignore").read().count("listening on") < 3:
    raise SystemExit("The local bridge to LiveKit did not start. See /content/bridge.log")
print("Network ready: LiveKit and n8n-on-LAN are reachable through 127.0.0.1.")


In [ ]:
# 1. Agent packages (main environment). A dependency warning about google-adk / opentelemetry is harmless.
!pip -q install livekit-agents livekit-plugins-silero livekit-plugins-openai livekit-api scipy httpx fastapi uvicorn uv
!pip freeze | grep -i -E "^(livekit|fastapi|uvicorn|httpx|scipy)" | tee /content/agent-requirements.txt


In [ ]:
# 2. Separate environments, one per program. Takes several minutes the first time.
def sh(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    out = (r.stdout + r.stderr).strip()
    if out: print(out[-1500:])
    return r.returncode

GPY = "/content/gemmaenv/bin/python"
if ENGINE == "vllm" and not os.path.exists(GPY): sh("uv venv /content/gemmaenv")
if ENGINE == "vllm":
    sh(f'uv pip install -q --python {GPY} "vllm[audio]"')
    ARCH = "from vllm import ModelRegistry; print([a for a in ModelRegistry.get_supported_archs() if 'Gemma4' in a])"
    known = subprocess.run([GPY, "-c", ARCH], capture_output=True, text=True).stdout
    if "Gemma4" not in known:
        print("This vLLM release does not know Gemma 4: installing the nightly build ...")
        sh(f'uv pip install -q --python {GPY} -U --pre "vllm[audio]" --extra-index-url https://wheels.vllm.ai/nightly')
        known = subprocess.run([GPY, "-c", ARCH], capture_output=True, text=True).stdout
    assert "Gemma4" in known, "vLLM does not list Gemma 4 (output above)."
    print("Gemma environment ready:", known.strip())
else:
    LLAMA_DIR, LSERVER = "/content/llama.cpp", "/content/llama.cpp/build/bin/llama-server"
    if not os.path.exists(LSERVER):
        cap = subprocess.run("nvidia-smi --query-gpu=compute_cap --format=csv,noheader", shell=True,
                             capture_output=True, text=True).stdout.split()[0].replace(".", "")
        env = dict(os.environ, PATH="/usr/local/cuda/bin:" + os.environ["PATH"])
        with open("/content/llama_build.log", "w") as blog:
            for cmd in (["git", "clone", "--depth", "1", "https://github.com/ggml-org/llama.cpp", LLAMA_DIR],
                        ["cmake", "-S", LLAMA_DIR, "-B", f"{LLAMA_DIR}/build", "-DGGML_CUDA=ON",
                         f"-DCMAKE_CUDA_ARCHITECTURES={cap}", "-DCMAKE_BUILD_TYPE=Release", "-DLLAMA_BUILD_TESTS=OFF"],
                        ["cmake", "--build", f"{LLAMA_DIR}/build", "--target", "llama-server", "-j", str(os.cpu_count() or 2)]):
                if cmd[0] == "git" and os.path.exists(LLAMA_DIR): continue
                print("running:", " ".join(cmd[:3]), "...")
                if subprocess.run(cmd, stdout=blog, stderr=subprocess.STDOUT, env=env).returncode:
                    print(open("/content/llama_build.log", errors="ignore").read()[-3000:])
                    raise SystemExit("llama.cpp build failed (log tail above).")
    print("llama.cpp built:", LSERVER, "(the model downloads in cell 4)")

if not os.path.exists("/content/venv/bin/python"): sh("uv venv /content/venv")
sh("uv pip install -q --python /content/venv/bin/python vllm")
sh('/content/venv/bin/python -c "import vllm; print(\'Mio vllm\', vllm.__version__)"')
if not os.path.exists("/content/mioenv/bin/python"): sh("uv venv /content/mioenv")
sh('uv pip install -q --python /content/mioenv/bin/python "git+https://github.com/Aratako/MioCodec" soundfile httpx numpy fastapi uvicorn huggingface_hub')
sh('/content/mioenv/bin/python -c "import miocodec, torch; print(\'miocodec ok | cuda\', torch.cuda.is_available())"')


In [ ]:
# 4. Gemma, ears + brain (vLLM or llama.cpp, from ENGINE in cell 0).
import time, urllib.request
def start_vllm(model, port, mem, log, extra=(), py="/content/venv/bin/python"):
    proc = subprocess.Popen(
        [py, "-m", "vllm.entrypoints.openai.api_server", "--model", model,
         "--gpu-memory-utilization", str(mem), "--port", str(port), *extra],
        stdout=open(log, "w"), stderr=subprocess.STDOUT, start_new_session=True,
        env=clean_env(PATH=os.path.dirname(py) + ":" + os.environ["PATH"], VLLM_USE_FLASHINFER_SAMPLER="0"))
    t0 = time.time()
    while True:
        try:
            urllib.request.urlopen(f"http://localhost:{port}/v1/models", timeout=2)
            print(f"{model} ready in {time.time()-t0:.0f}s"); return proc
        except Exception:
            if proc.poll() is not None:
                print(open(log).read()[-3000:]); raise SystemExit(f"vLLM for {model} stopped. Log above.")
            if time.time() - t0 > 900:
                raise SystemExit(f"vLLM for {model} not ready after 15 minutes. See {log}")
            time.sleep(3)

def stop_vllm(port=None):
    import signal
    pat = "vllm[.]entrypoints" + (f".*--port {port}" if port else "")
    for pid in subprocess.run(["pgrep", "-f", pat], capture_output=True, text=True).stdout.split():
        try:
            if os.getpgid(int(pid)) == int(pid) != os.getpgid(0):
                os.killpg(int(pid), signal.SIGKILL)
            else:
                os.kill(int(pid), signal.SIGKILL)
        except Exception: pass
    time.sleep(2)
    gpu = subprocess.run("nvidia-smi --query-compute-apps=pid --format=csv,noheader", shell=True, capture_output=True, text=True).stdout.split()
    for pid in gpu:
        info = subprocess.run(["ps", "-o", "ppid=,args=", "-p", pid], capture_output=True, text=True).stdout.split(None, 1)
        if len(info) == 2 and info[0] == "1" and "VLLM" in info[1]:
            subprocess.run(["kill", "-9", pid]); print("removed leftover worker, pid", pid)
    time.sleep(2)

stop_vllm(8000); subprocess.run(["pkill", "-9", "-f", "llama.cpp/build/bin/llama-server"]); time.sleep(2)
if ENGINE == "vllm":
    LLM_MODEL = GEMMA_MODEL
    vllm_proc = start_vllm(GEMMA_MODEL, 8000, 0.3, "/content/vllm.log",
                           ["--max-model-len", "4096", "--max-num-seqs", "3",
                            "--limit-mm-per-prompt", '{"image": 0, "audio": 1}'], py="/content/gemmaenv/bin/python")
else:
    LLM_MODEL = "gemma"
    LSERVER = "/content/llama.cpp/build/bin/llama-server"
    from huggingface_hub import hf_hub_download, list_repo_files
    def _pick(repo, pattern, mm):
        fs = [f for f in list_repo_files(repo) if f.endswith(".gguf") and (("mmproj" in f.lower()) == mm)]
        fs = [f for f in fs if pattern.lower() in f.lower()] or ([] if not mm else fs)
        order = ["f16", "bf16", "f32", "q8"]
        return sorted(fs, key=lambda f: (next((i for i, q in enumerate(order) if q in f.lower()), 9) if mm else len(f)))
    g = _pick(GGUF_REPO, GGUF_QUANT, False)
    m = _pick(GGUF_REPO, "", True) or _pick("ggml-org/gemma-4-E4B-it-GGUF", "", True)
    MM_REPO = GGUF_REPO if _pick(GGUF_REPO, "", True) else "ggml-org/gemma-4-E4B-it-GGUF"
    GGUF_PATH = hf_hub_download(GGUF_REPO, g[0], local_dir="/content/gguf")
    MMPROJ_PATH = hf_hub_download(MM_REPO, m[0], local_dir="/content/gguf")
    lproc = subprocess.Popen([LSERVER, "-m", GGUF_PATH, "--mmproj", MMPROJ_PATH, "--host", "127.0.0.1", "--port", "8000",
                              "-ngl", "99", "-np", "3", "-c", str(4096 * 3), "-fa", "on", "--jinja", "--no-webui"],
                             stdout=open("/content/llama.log", "w"), stderr=subprocess.STDOUT, start_new_session=True)
    t0 = time.time()
    while True:
        try:
            urllib.request.urlopen("http://localhost:8000/health", timeout=2); break
        except Exception:
            if lproc.poll() is not None or time.time() - t0 > 600:
                print(open("/content/llama.log", errors="ignore").read()[-3000:]); raise SystemExit("llama-server failed.")
            time.sleep(2)
    print(f"Gemma (llama.cpp) ready in {time.time()-t0:.0f}s")


In [ ]:
# 5. Mio voice service: its speech model (vLLM) plus the codec service that turns it into audio.
def wait_ready(proc, log, marker, name, limit=600):
    t0 = time.time()
    while True:
        text = open(log, errors="ignore").read()
        if marker in text:
            print("\n".join(l for l in text.splitlines() if l.startswith(marker.split("]")[0] + "]"))); return
        if proc.poll() is not None:
            print(text[-3000:]); raise SystemExit(f"{name} stopped. Log above.")
        if time.time() - t0 > limit:
            raise SystemExit(f"{name} not ready after {limit // 60} minutes.")
        time.sleep(3)

subprocess.run("pkill -9 -f mio_server[.]py", shell=True); stop_vllm(8002)
mio_vllm = start_vllm("SPRINGLab/Indic-Mio", 8002, 0.15, "/content/mio_vllm.log",
                      ["--max-model-len", "1024", "--max-num-seqs", "2"])
mio_proc = subprocess.Popen(["/content/mioenv/bin/python", "-u", f"{SRC}/mio_server.py"],
                            stdout=open("/content/mio.log", "w"), stderr=subprocess.STDOUT,
                            env=clean_env(MIO_REF_WAV=REF_VOICE, MIO_SPEED=str(SPEAKING_RATE),
                                          MIO_PAUSE=str(CLAUSE_PAUSE), MIO_STYLE_HI=STYLE_HINDI,
                                          MIO_STYLE_EN=STYLE_ENGLISH), start_new_session=True)
wait_ready(mio_proc, "/content/mio.log", "[mio] ready", "Mio voice service")


In [ ]:
# 5b. Her instructions live in src/prompts/sys_prompt.txt. Edit that file in the repo (or in Colab's file browser),
# then restart the agent in the services cell to apply.
PROMPT_FILE = f"{SRC}/prompts/sys_prompt.txt"
os.environ["SYS_PROMPT_FILE"] = PROMPT_FILE
print(open(PROMPT_FILE, encoding="utf-8").read())


In [ ]:
# 5c. Try her instructions by typing (no call needed; needs cell 4 running).
TRY = [
    "Hello, M.A. Creations se baat ho rahi hai?",
    "Rate kya hai ek puzzle ka? WhatsApp pe catalogue bhej do.",
    "Mujhe 300 wooden box chahiye, kitne din mein milega?",
    "Mujhe aapke manager se baat karni hai.",
    "Aapka office kahan hai?",
    "Pura address batao.",
    "Do you make wooden toys? Send me the catalog link on WhatsApp.",
]
import httpx, re, time
SYS_NOW = open(PROMPT_FILE, encoding="utf-8").read()
for line in TRY:
    t0 = time.time()
    j = httpx.post("http://localhost:8000/v1/chat/completions", timeout=60, json={
        "model": LLM_MODEL, "max_tokens": 150, "temperature": 0.3, "top_p": 0.95,
        "chat_template_kwargs": {"enable_thinking": False},
        "messages": [{"role": "system", "content": SYS_NOW}, {"role": "user", "content": line}]}).json()
    reply = j["choices"][0]["message"]["content"].strip()
    print(f"CALLER: {line}\nSHE   : {reply}   [{time.time() - t0:.2f}s]\n")


In [ ]:
# 6. Start LiveKit Agent Worker + Outbound Dialer Service (:8060) + Aegis Dashboard (:8050) via liblk_redirect.so
AGENT_NAME = "aegis-agent"
subprocess.run("pkill -9 -f live_agent[.]py; pkill -9 -f dialer_service[.]py; pkill -9 -f dashboard_backend[.]py", shell=True)
time.sleep(2)

if not os.path.exists("/content/liblk_redirect.so"):
    raise SystemExit("Run the network cell first: it builds the Tailscale bridge and /content/liblk_redirect.so.")
AGENT_ENV = dict(
    os.environ,
    LLM_MODEL=LLM_MODEL,
    AGENT_NAME=AGENT_NAME,
    LD_PRELOAD="/content/liblk_redirect.so",
)

# 1. Start LiveKit Agent Worker (routed over native TCP bridge + liblk_redirect.so)
agent_proc = subprocess.Popen(
    ["python3", "-u", f"{SRC}/live_agent.py", "start"],
    stdout=open("/content/agent.log", "w"),
    stderr=subprocess.STDOUT,
    cwd="/content",
    env=AGENT_ENV,
    start_new_session=True,
)

# 2. Start Outbound Dialer API on port 8060
dialer_proc = subprocess.Popen(
    ["python3", "-u", f"{SRC}/dialer_service.py"],
    stdout=open("/content/dialer.log", "w"),
    stderr=subprocess.STDOUT,
    cwd="/content",
    env=AGENT_ENV,
    start_new_session=True,
)

# 3. Start Aegis Live Dashboard on port 8050
dash_proc = subprocess.Popen(
    ["python3", "-u", f"{SRC}/dashboard_backend.py"],
    stdout=open("/content/dashboard.log", "w"),
    stderr=subprocess.STDOUT,
    cwd="/content",
    start_new_session=True,
)

t0 = time.time()
while True:
    log = open("/content/agent.log", errors="ignore").read()
    if "registered worker" in log:
        print(f"Agent '{AGENT_NAME}' ready in {time.time()-t0:.0f}s and connected to {os.environ['LIVEKIT_URL']}!")
        break
    if agent_proc.poll() is not None or "worker failed" in log:
        print(log[-3000:]); raise SystemExit("Agent stopped. The log tail is above.")
    if time.time() - t0 > 180:
        print(log[-3000:]); raise SystemExit("Agent not ready after 3 minutes. See /content/agent.log")
    time.sleep(2)

# Dialer health, then expose :8060 to your tailnet so n8n can call POST /dial
import json, urllib.request
for _ in range(30):
    try:
        h = json.loads(urllib.request.urlopen("http://127.0.0.1:8060/health", timeout=2).read())
        break
    except Exception:
        time.sleep(1)
else:
    print(open("/content/dialer.log", errors="ignore").read()[-2000:]); raise SystemExit("Dialer did not start. Log above.")
print("Dialer:", h)
r = subprocess.run(["tailscale", "serve", "--bg", "--tcp=8060", "tcp://localhost:8060"], capture_output=True, text=True, timeout=30)
print((r.stdout + r.stderr).strip()[-400:])
ts_ip = subprocess.run(["tailscale", "ip", "-4"], capture_output=True, text=True).stdout.strip()
print(f"- n8n:     POST http://{ts_ip}:8060/dial   header X-Api-Key = your DIALER_API_KEY   (allow this node on port 8060 in your Tailscale ACL)")
print("- Inbound: dial 8000 from any office landline.")
print("- Test:    run Cell 7 below, or open the dashboard (Cell 7a).")


## n8n contract
Full request/response JSON, error codes and workflow shape are in the repository README (section *API contract*). In short: n8n calls `POST /dial` with `job_id` and `callback_url`, gets `202` immediately, and receives the outcome plus transcript on `callback_url` when the call ends. The `/dial` endpoint is idempotent by `job_id`.


In [ ]:
# 7. Place a test call: POST /dial with the API key, then wait for the outcome callback.
import httpx, json, os, threading, time, uuid
from http.server import BaseHTTPRequestHandler, HTTPServer

TARGET_PHONE = secret("TEST_PHONE", required=False)   # your own number; set the TEST_PHONE secret
CUSTOMER_NAME = "Test Caller"
TWENTY_CRM_ID = ""
if not TARGET_PHONE:
    raise SystemExit("Set the TEST_PHONE Colab secret to a number you own.")

if "RECEIVED" not in globals():
    RECEIVED = []

class _Sink(BaseHTTPRequestHandler):
    def do_POST(self):
        n = int(self.headers.get("Content-Length", 0))
        try:
            RECEIVED.append(json.loads(self.rfile.read(n) or b"{}"))
        except Exception:
            RECEIVED.append({"error": "bad json"})
        self.send_response(200)
        self.end_headers()
        self.wfile.write(b"ok")
    def log_message(self, *a):
        pass

try:
    threading.Thread(target=HTTPServer(("127.0.0.1", 8070), _Sink).serve_forever, daemon=True).start()
except OSError:
    pass

job_id = f"test-{uuid.uuid4().hex[:8]}"

with httpx.Client(trust_env=False, timeout=25.0) as client:
    r = client.post(
        "http://127.0.0.1:8060/dial",
        headers={"X-Api-Key": os.environ["DIALER_API_KEY"]},
        json={
            "phone": TARGET_PHONE,
            "customer_name": CUSTOMER_NAME,
            "twenty_person_id": TWENTY_CRM_ID,
            "job_id": job_id,
            "callback_url": "http://127.0.0.1:8070/callback"
        }
    )

try:
    resp_data = r.json()
except ValueError:
    resp_data = {"raw_text": r.text}

print("Dial response:", r.status_code, resp_data)

t0 = time.time()
while r.status_code in (200, 202) and (time.time() - t0) < 300 and not any(x.get("job_id") == job_id for x in RECEIVED):
    time.sleep(2)

for x in RECEIVED:
    if x.get("job_id") == job_id:
        print("\nOutcome callback:", json.dumps({k: v for k, v in x.items() if k != "transcript"}, indent=2, ensure_ascii=False))
        print("\nTranscript:\n" + x.get("transcript", ""))

In [ ]:
# 7a. Open the Live Aegis Call Cockpit inside Colab (or via public tunnel)
from google.colab import output
print("Opening Aegis Voice Cockpit on port 8050...")
output.serve_kernel_port_as_window(8050)
output.serve_kernel_port_as_iframe(8050, height="680")


In [ ]:
# 7b. Tune her pace DURING a call: change the values, run this cell, and hear it from her next sentence.
rate  = 0.95     # 1.0 = as Mio generates it; 0.9 = slower (range 0.6-1.4)
pause = 0.15     # seconds of silence between clauses (0 - 1.0)
style_hi, style_en = "happy", "happy"
import json, urllib.request
body = json.dumps(dict(speed=rate, pause=pause, style_hi=style_hi, style_en=style_en)).encode()
r = urllib.request.Request("http://127.0.0.1:8003/config", data=body, headers={"Content-Type": "application/json"})
print(json.loads(urllib.request.urlopen(r, timeout=5).read()))


In [ ]:
# 8. What happened on the call: what she heard, how long each stage took, n8n webhooks, and GPU memory.
!grep -a -E "^\[(call|heard|mio|turn|llm |tts |load|n8n)|skipped|Error" /content/agent.log | tail -50
print("\n--- Dialer Service Log ---")
!tail -15 /content/dialer.log
print("\n--- GPU Memory per program (MiB) ---")
def args_of(pid):
    return subprocess.run(f"ps -o args= -p {pid}", shell=True, capture_output=True, text=True).stdout.strip()
apps = subprocess.run("nvidia-smi --query-compute-apps=pid,used_memory --format=csv,noheader,nounits", shell=True, capture_output=True, text=True).stdout
for line in apps.strip().splitlines():
    pid, mem = [x.strip() for x in line.split(",")]
    cmd = args_of(pid)
    if "vllm.entrypoints" not in cmd and "VLLM" in cmd:
        ppid = subprocess.run(f"ps -o ppid= -p {pid}", shell=True, capture_output=True, text=True).stdout.strip()
        cmd = "LEFTOVER model worker: rerun cell 4 to remove it" if ppid == "1" else args_of(ppid) if ppid else cmd
    name = (cmd if cmd.startswith("LEFTOVER") else "Indic-Mio speech model (vLLM)" if "Indic-Mio" in cmd
            else "Gemma ears + brain (llama.cpp)" if "llama-server" in cmd
            else "Gemma ears + brain (vLLM)" if "gemma-4" in cmd
            else "Mio voice service (codec)" if "mio_server" in cmd
            else "agent" if "live_agent" in cmd else cmd[:60] or "unknown")
    print(f"  {int(mem):6d}  {name}")
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv
print("\n--- Outcomes that could not be delivered to n8n (replay these) ---")
!ls -1 /content/outbox 2>/dev/null || echo none


In [ ]:
# 9. Listen to what she said (latest clips)
from IPython.display import Audio, display
for f in sorted(glob.glob("/content/tts_debug/*.wav"))[-5:]:
    print(f); display(Audio(filename=f))


In [ ]:
# 10. Stop everything. Then Runtime -> Disconnect and delete runtime, so idle time doesn't use compute units.
subprocess.run("pkill -9 -f live_agent[.]py; pkill -9 -f dialer_service[.]py; pkill -9 -f dashboard_backend[.]py; pkill -9 -f mio_server[.]py; pkill -9 -f ts_bridge[.]py; pkill -9 -f llama.cpp/build/bin/llama-server", shell=True)
stop_vllm()
subprocess.run(["tailscale", "serve", "reset"])
!nvidia-smi --query-gpu=memory.used --format=csv
